# 4. Spectral Analysis of Time Series

Spectral analysis studies time series in the **frequency domain**. Instead of asking "what happens at time $t$?", we ask "how much energy is at frequency $\omega$?".

This notebook covers:
- The raw periodogram
- Smoothed periodogram (Welch's method)
- Theoretical spectrum of an AR(2) process

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import periodogram, welch

%matplotlib inline
np.random.seed(42)

## 4.1 Simulating an AR(2) Process

The AR(2) process $X_t = \phi_1 X_{t-1} + \phi_2 X_{t-2} + \varepsilon_t$ exhibits a spectral peak at a frequency determined by its coefficients.

In [ ]:
n = 1024
eps = np.random.normal(0, 1, n)
x = np.zeros(n)
phi1, phi2 = 1.5, -0.85

for t in range(2, n):
    x[t] = phi1*x[t-1] + phi2*x[t-2] + eps[t]

plt.figure(figsize=(12, 3))
plt.plot(x, lw=0.5)
plt.title('AR(2) Process')
plt.xlabel('t')
plt.show()

## 4.2 Raw Periodogram

The **periodogram** estimates the power spectral density (PSD) as $I(\omega) = \frac{1}{n}|\sum_t x_t e^{-i\omega t}|^2$. It is an inconsistent estimator (noisy).

In [ ]:
freq_p, Pxx_p = periodogram(x, fs=1, scaling='density')

plt.figure(figsize=(10, 4))
plt.semilogy(freq_p, Pxx_p, lw=0.5)
plt.title('Raw Periodogram')
plt.xlabel('Frequency')
plt.ylabel('PSD')
plt.grid(True, alpha=0.3)
plt.show()

## 4.3 Smoothed Periodogram (Welch's Method)

**Welch's method** averages periodograms over overlapping segments, producing a smoother and more consistent estimate of the PSD.

In [ ]:
freq_w, Pxx_w = welch(x, fs=1, nperseg=256, scaling='density')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].semilogy(freq_p, Pxx_p, lw=0.5)
axes[0].set_title('Raw Periodogram')
axes[0].set_xlabel('Frequency')

axes[1].semilogy(freq_w, Pxx_w, lw=1.5, color='red')
axes[1].set_title('Smoothed Periodogram (Welch)')
axes[1].set_xlabel('Frequency')
plt.tight_layout()
plt.show()

## 4.4 Theoretical AR(2) Spectrum

For an AR(2), the theoretical spectrum is:
$$f(\omega) = \frac{\sigma^2}{2\pi |1 - \phi_1 e^{-i\omega} - \phi_2 e^{-2i\omega}|^2}$$

We can compare this exact formula with the empirical estimate.

In [ ]:
omega = np.linspace(0.01, np.pi, 500)
phi_z = 1 - phi1*np.exp(-1j*omega) - phi2*np.exp(-2j*omega)
f_theo = 1 / (2*np.pi * np.abs(phi_z)**2)

plt.figure(figsize=(8, 4))
plt.plot(omega, f_theo, 'b-', label='Theoretical', lw=2)
plt.title('Theoretical AR(2) Spectrum')
plt.xlabel('$\\omega$')
plt.ylabel('$f(\\omega)$')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Key Takeaways

- The **periodogram** decomposes a time series into frequency components
- The raw periodogram is **noisy**; use **Welch's method** for smoother estimates
- AR processes have a **parametric spectral form** that we can compute exactly
- Spectral peaks indicate **dominant periodicities** in the data